In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline
import joblib
import os
import lime
import lime.lime_text
from lime.lime_text import LimeTextExplainer
import shap

%matplotlib inline

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


## Step 01: Define complaint classification objective

The objective of this project is to classify customer complaints into 5 specific categories (Billing, Technical Support, Service Quality, Delivery, Account) to improve the prioritization of resolution and identify systemic service issues.

In [ ]:
print("Project Title: Customer Complaint Classification and Resolution Insights")
print("Objective: Classify customer complaints into 5 categories to improve resolution prioritization and identify systemic service issues.")


## Step 02: Collect customer complaint datasets

We will load the CSV dataset using pandas and explore the initial structure of the data.

In [ ]:
df = pd.read_csv('../data/complaints.csv')
display(df.head())
print("\nDataset Shape:", df.shape)
print("\nDataset Info:")
df.info()


## Step 03: Understand complaint categories and priorities

Let's look at the unique categories and priority levels, and display the counts for each to understand our data distribution.

In [ ]:
print("Unique Categories:", df['category'].unique())
print("Unique Priorities:", df['priority'].unique())

print("\nCategory Value Counts:")
print(df['category'].value_counts())

print("\nPriority Value Counts:")
print(df['priority'].value_counts())

print("\nCross-tabulation of Category vs Priority:")
display(pd.crosstab(df['category'], df['priority']))


## Step 04: Preprocess and normalize text data

We need to clean the text data before feature extraction. This involves:
- Lowercasing the text
- Removing special characters, numbers, and extra whitespace
- Tokenization and stopwords removal
- Lemmatization

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(cleaned_tokens)

df['cleaned_text'] = df['complaint_text'].apply(clean_text)
display(df[['complaint_text', 'cleaned_text']].head())


## Step 05: Perform exploratory analysis on complaint distribution

Visualizing the distribution of categories, priorities, common words, and complaints over time.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='category', order=df['category'].value_counts().index, palette='viridis')
plt.title('Category Distribution')
plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='priority', order=df['priority'].value_counts().index, palette='magma')
plt.title('Priority Distribution')
plt.show()

all_text = ' '.join(df['cleaned_text'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of All Complaint Text')
plt.show()

try:
    df['date_received'] = pd.to_datetime(df['date_received'])
    df_time = df.groupby(df['date_received'].dt.to_period('M')).size()
    df_time.plot(kind='line', figsize=(10, 5), title='Complaints Over Time', marker='o')
    plt.ylabel('Number of Complaints')
    plt.show()
except Exception as e:
    print("Could not plot timeline:", e)

crosstab = pd.crosstab(df['category'], df['priority'])
plt.figure(figsize=(8, 6))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues')
plt.title('Category vs Priority Heatmap')
plt.show()


## Step 06: Label complaint categories

Encode the categorical labels into numeric values for our machine learning model.

In [ ]:
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Category Mapping:", mapping)
display(df[['category', 'category_encoded']].head(10))


## Step 07: Engineer text features

Apply TF-IDF vectorization to represent our text data numerically.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_text'])
print("Feature matrix shape:", X.shape)
print("Top 20 features:", tfidf.get_feature_names_out()[:20])


## Step 08: Split data into training and testing sets

Split the data using an 80/20 ratio with stratification to ensure balanced classes in both sets.

In [ ]:
y = df['category_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])


## Step 09: Train classification model

Train a Logistic Regression model and a Random Forest classifier as a comparison.

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
print("Models trained successfully.")


## Step 10: Evaluate classification accuracy

Evaluate both models using accuracy score, classification report, and a confusion matrix heatmap.

In [ ]:
lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))

print("\nClassification Report (Logistic Regression):")
print(classification_report(y_test, lr_preds, target_names=le.classes_))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, lr_preds), annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix (Logistic Regression)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

best_model = lr_model  # Selecting Logistic Regression as our best model based on typical text baseline performance
print("Selected Best Model: Logistic Regression")


## Step 11: Identify frequently occurring issues

Analyze common categories and identify top keywords driving each category using feature importance (coefficients from our logistic regression model).

In [ ]:
common_categories = df['category'].value_counts()
print("Most common complaint categories:\n", common_categories)

plt.figure(figsize=(10, 5))
common_categories.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Top Complaint Issues by Category')
plt.ylabel('Count')
plt.show()

feature_names = tfidf.get_feature_names_out()
for idx, class_name in enumerate(le.classes_):
    coefs = lr_model.coef_[idx]
    top_indices = coefs.argsort()[-5:][::-1]
    top_keywords = [feature_names[i] for i in top_indices]
    print(f"Top keywords for {class_name}: {', '.join(top_keywords)}")


## Step 12: Generate resolution priority logic

Create a prioritization score based on model confidence and category to better handle incoming complaints.

In [ ]:
def priority_scoring(probabilities, predicted_classes):
    priorities = []
    for prob, cls in zip(probabilities, predicted_classes):
        confidence = max(prob)
        if confidence > 0.8:
            priorities.append('High')
        elif confidence > 0.5:
            priorities.append('Medium')
        else:
            priorities.append('Low')
    return priorities

probs = best_model.predict_proba(X_test)
test_priorities = priority_scoring(probs, lr_preds)

priority_df = pd.DataFrame({'Predicted Category': le.inverse_transform(lr_preds), 'Priority': test_priorities})
display(priority_df.head(10))

plt.figure(figsize=(8, 4))
sns.countplot(data=priority_df, x='Priority', order=['High', 'Medium', 'Low'], palette='Set2')
plt.title('Resolution Priority Distribution on Test Set')
plt.show()


## Explainable AI (SHAP/LIME)

Explainability analysis using LIME for local text explanation and SHAP for global/local feature importance.

In [ ]:
explainer = LimeTextExplainer(class_names=list(le.classes_))
pipeline = make_pipeline(tfidf, best_model)

sample_idx = 0
sample_text = df['cleaned_text'].iloc[sample_idx]

print("Sample Text for LIME:", sample_text)
exp = explainer.explain_instance(sample_text, pipeline.predict_proba, num_features=6, top_labels=1)
exp.show_in_notebook(text=True)

# SHAP analysis
try:
    explainer_shap = shap.LinearExplainer(best_model, X_train, feature_perturbation="interventional")
    shap_values = explainer_shap.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, class_names=list(le.classes_))
except Exception as e:
    print("SHAP analysis encountered an error:", e)


## Step 13: Save NLP model/vectorizer

Save the best trained model, TF-IDF vectorizer, and LabelEncoder for deployment.

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/complaint_classifier.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

loaded_model = joblib.load('../models/complaint_classifier.pkl')
print("Model and components successfully saved to the '../models' directory and verified.")


## Step 14: (Reference to Streamlit app)

The application source code is provided in `app.py` in the root or `app/` directory, which loads the saved model components to provide a web interface.

## Step 15: Test multiple text cases

Define a few new, unseen test cases to see how well our model works.

In [ ]:
test_cases = [
    "My internet connection drops every hour and I cannot work.",
    "I have been overcharged on my last bill by $50.",
    "The customer service agent was very rude and unhelpful.",
    "My replacement router never arrived as promised.",
    "I can't log into my account portal, it keeps saying wrong password."
]

test_cases_cleaned = [clean_text(text) for text in test_cases]
test_vec = tfidf.transform(test_cases_cleaned)
predictions = loaded_model.predict(test_vec)
probabilities = loaded_model.predict_proba(test_vec)

results = []
for i, text in enumerate(test_cases):
    pred_cat = le.inverse_transform([predictions[i]])[0]
    conf = max(probabilities[i])
    results.append({'Complaint': text, 'Predicted Category': pred_cat, 'Confidence': f"{conf:.2f}"})

results_df = pd.DataFrame(results)
display(results_df)


## Step 16: Deploy app on Streamlit Cloud (reference)

To deploy this application to Streamlit Cloud:
1. Ensure `app.py`, `requirements.txt`, and the `models/` directory are committed to a GitHub repository.
2. Log in to [Streamlit Community Cloud](https://share.streamlit.io/).
3. Click 'New app', point it to your GitHub repository and specify `app.py` as the main file path.
4. Add any required configurations in `.streamlit/config.toml` if necessary.
5. Click Deploy.

## Step 17: Document operational recommendations

### Operational Recommendations
- **Resource Allocation:** Allocate more support staff to the most frequent complaint categories identified in Step 11.
- **Priority Handling:** Implement an automated routing system using the priority logic from Step 12 to ensure 'High' priority (high confidence) cases are resolved immediately.
- **Root Cause Analysis:** Regularly review the top keywords extracted for 'Service Quality' and 'Technical Support' to identify and fix systemic issues (e.g., repeating network outages).
- **Continuous Improvement:** Periodically retrain the NLP model with new data to keep up with evolving customer language and new product lines.